# 구조화된 출력 (Structured Output)

구조화된 출력을 사용하면 에이전트가 특정하고 예측 가능한 형식으로 데이터를 반환할 수 있습니다. 자연어 응답을 구문 분석하는 대신 애플리케이션에서 직접 사용할 수 있는 JSON 객체, Pydantic 모델 또는 데이터클래스 형태로 구조화된 데이터를 얻을 수 있습니다.

**구조화된 출력 전략:**

| 전략 | 설명 |
|:---|:---|
| **ProviderStrategy** | OpenAI, Anthropic, Grok 등 네이티브 구조화된 출력 지원 모델용 |
| **ToolStrategy** | 도구 호출을 통한 구조화된 출력 (대부분의 모델 지원) |
| **자동 선택** | 스키마 타입만 전달 시 모델에 따라 최적 전략 자동 선택 |

LangChain의 `create_agent`는 `response_format` 매개변수로 구조화된 출력을 설정하며, 결과는 에이전트 상태의 `structured_response` 키에 반환됩니다.

> 📖 **참고 문서**: [LangChain Structured Output](https://docs.langchain.com/oss/python/langchain/structured-output)

## 환경 설정

구조화된 출력 튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`를 사용하여 API 키를 로드하고, `langchain_teddynote`의 로깅 기능을 활성화하여 LangSmith에서 실행 추적을 확인할 수 있도록 합니다.

LangSmith 추적을 활성화하면 에이전트의 구조화된 출력 생성 과정을 시각적으로 디버깅할 수 있어, 스키마 검증 오류나 재시도 과정을 파악하는 데 도움이 됩니다.

아래 코드는 환경 변수를 로드하고 LangSmith 프로젝트를 설정합니다.

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)

# LangSmith 추적 설정
logging.langsmith("LangGraph-V1-Tutorial")

LangSmith 추적을 시작합니다.
[프로젝트명]
LangGraph-V1-Tutorial


---

## Response Format

에이전트가 구조화된 데이터를 반환하는 방법을 `response_format` 매개변수로 제어합니다:

| 설정 | 설명 |
|:---|:---|
| **ToolStrategy[T]** | 도구 호출을 통한 구조화된 출력 |
| **ProviderStrategy[T]** | 제공자 네이티브 구조화된 출력 사용 |
| **type[T]** | 스키마 타입 직접 전달 - 모델에 따라 최적 전략 자동 선택 |
| **None** | 구조화된 출력 없음 |

스키마 타입이 직접 제공되면 LangChain이 자동으로 최적의 전략을 선택합니다:
- 네이티브 구조화된 출력 지원 모델(OpenAI, Anthropic, Grok)에는 `ProviderStrategy`
- 다른 모든 모델에는 `ToolStrategy`

구조화된 응답은 에이전트의 최종 상태의 `structured_response` 키에 반환됩니다.

---

## Provider Strategy

일부 모델 제공자(OpenAI, Anthropic, Grok 등)는 API를 통해 구조화된 출력을 네이티브로 지원합니다. 이 방법은 모델이 JSON 스키마를 직접 이해하고 준수하도록 강제하므로, 사용 가능한 경우 가장 신뢰할 수 있는 방법입니다.

스키마 타입을 `create_agent`의 `response_format`에 직접 전달하면, LangChain이 지원 모델에 대해 자동으로 `ProviderStrategy`를 사용합니다. 지원되지 않는 모델의 경우 자동으로 `ToolStrategy`로 폴백됩니다.

> 📖 **참고 문서**: [LangChain Models](https://docs.langchain.com/oss/python/langchain/models.md)

### Pydantic 모델

Pydantic 모델을 사용하면 필드에 대한 상세한 설명과 검증 규칙을 정의할 수 있습니다. `Field`의 `description`은 모델이 각 필드의 용도를 이해하는 데 도움이 되며, 이를 통해 더 정확한 구조화된 출력을 생성할 수 있습니다.

Pydantic은 타입 검증, 기본값 설정, 범위 제한(`ge`, `le`) 등 풍부한 검증 기능을 제공하므로, 복잡한 스키마 정의에 가장 적합합니다.

아래 코드는 Pydantic 모델로 연락처 정보를 추출하는 예시입니다.

In [2]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """사람의 연락처 정보를 나타내는 스키마

    이름, 이메일, 전화번호를 구조화된 형태로 추출합니다.
    """

    name: str = Field(description="The name of the person")  # 이름
    email: str = Field(description="The email address of the person")  # 이메일 주소
    phone: str = Field(description="The phone number of the person")  # 전화번호


# 모델 초기화
# OpenAI 키 사용 시 gpt-4.1-mini, gpt-5.2 등으로 변경하세요.
model = init_chat_model("claude-sonnet-4-5")

# 에이전트 생성 - response_format에 스키마 타입 전달 시 ProviderStrategy 자동 선택
agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo,
)

# 에이전트 실행
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

# 구조화된 응답 확인
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


### 데이터클래스

Python의 `@dataclass` 데코레이터를 사용하여 스키마를 정의할 수도 있습니다. 데이터클래스는 Pydantic보다 가볍고 Python 표준 라이브러리의 일부이므로 추가 의존성 없이 사용할 수 있습니다.

필드 설명은 주석(comment)으로 추가하며, 모델이 이를 참고하여 각 필드에 적절한 값을 채웁니다.

아래 코드는 데이터클래스로 연락처 정보를 추출하는 예시입니다.

In [3]:
from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """사람의 연락처 정보를 나타내는 스키마"""

    name: str  # 이름 (The name of the person)
    email: str  # 이메일 주소 (The email address of the person)
    phone: str  # 전화번호 (The phone number of the person)


# 에이전트 생성 - 위에서 초기화한 model 재사용
agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo,  # ProviderStrategy 자동 선택
)

# 에이전트 실행
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: Jane Smith, jane@example.com, (555) 987-6543",
            }
        ]
    }
)

# 구조화된 응답 확인
print(result["structured_response"])

ContactInfo(name='Jane Smith', email='jane@example.com', phone='(555) 987-6543')


### TypedDict

`TypedDict`를 사용하면 딕셔너리 형태로 구조화된 출력을 받을 수 있습니다. 반환값이 Python 딕셔너리이므로 JSON 직렬화가 용이하고, API 응답이나 데이터베이스 저장에 바로 활용할 수 있습니다.

Pydantic이나 데이터클래스와 달리 인스턴스가 아닌 딕셔너리로 반환되므로, 후처리가 간단한 경우에 적합합니다.

아래 코드는 TypedDict로 연락처 정보를 추출하는 예시입니다.

In [4]:
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """사람의 연락처 정보를 나타내는 스키마"""

    name: str  # 이름 (The name of the person)
    email: str  # 이메일 주소 (The email address of the person)
    phone: str  # 전화번호 (The phone number of the person)


# 에이전트 생성
agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo,  # ProviderStrategy 자동 선택
)

# 에이전트 실행
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: Teddy Lee, teddy@example.com, (555) 111-2222",
            }
        ]
    }
)

# 구조화된 응답 확인 - 딕셔너리 형태로 반환됨
print(result["structured_response"])

{'name': 'Teddy Lee', 'email': 'teddy@example.com', 'phone': '(555) 111-2222'}


---

## Tool Calling Strategy

네이티브 구조화된 출력을 지원하지 않는 모델의 경우, LangChain은 도구 호출(Tool Calling)을 사용하여 동일한 결과를 달성합니다. `ToolStrategy`는 도구 호출을 지원하는 대부분의 최신 모델에서 작동하므로 범용성이 높습니다.

`ToolStrategy`를 명시적으로 사용하면 네이티브 지원 여부와 관계없이 항상 도구 호출 방식을 사용합니다. 이는 일관된 동작이 필요하거나 특정 모델에서 네이티브 방식에 문제가 있을 때 유용합니다.

> 📖 **참고 문서**: [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)

### Pydantic 모델

`ToolStrategy`를 사용할 때도 Pydantic 모델의 필드 설명과 검증 규칙이 동일하게 적용됩니다. `Literal` 타입으로 허용값을 제한하고, `ge`(greater than or equal), `le`(less than or equal) 등의 검증자로 범위를 지정할 수 있습니다.

스키마 검증에 실패하면 에이전트가 자동으로 재시도하여 올바른 형식의 출력을 생성하도록 유도합니다.

아래 코드는 ToolStrategy로 제품 리뷰를 분석하는 예시입니다.

In [5]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """제품 리뷰 분석 결과를 나타내는 스키마"""

    rating: int | None = Field(
        description="The rating of the product", ge=1, le=5
    )  # 평점 (1-5)
    sentiment: Literal["positive", "negative"] = Field(
        description="The sentiment of the review"
    )  # 감정 분석 결과
    key_points: list[str] = Field(
        description="The key points of the review. Lowercase, 1-3 words each."
    )  # 핵심 포인트


# 에이전트 생성 - ToolStrategy 명시적 사용
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(ProductReview),
)

# 에이전트 실행
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'",
            }
        ]
    }
)

# 구조화된 응답 확인
print(result["structured_response"])

rating=5 sentiment='positive' key_points=['fast shipping', 'expensive', 'great product']


### Union 타입

`Union` 타입을 사용하여 여러 스키마 옵션을 제공할 수 있습니다. 모델은 입력 컨텍스트에 따라 가장 적절한 스키마를 자동으로 선택하므로, 다양한 유형의 입력을 하나의 에이전트로 처리할 수 있습니다.

예를 들어, 고객 피드백을 처리할 때 긍정적인 리뷰와 불만 사항을 서로 다른 스키마로 분류하여 각각에 맞는 후속 처리를 할 수 있습니다.

아래 코드는 리뷰와 불만을 구분하여 분석하는 예시입니다.

In [6]:
from pydantic import BaseModel, Field
from typing import Literal, Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """제품 리뷰 분석 결과를 나타내는 스키마
    
    긍정/부정 감정 분석과 핵심 포인트를 추출합니다.
    """

    rating: int | None = Field(
        description="The rating of the product", ge=1, le=5
    )  # 평점 (1-5)
    sentiment: Literal["positive", "negative"] = Field(
        description="The sentiment of the review"
    )  # 감정 분석 결과
    key_points: list[str] = Field(
        description="The key points of the review. Lowercase, 1-3 words each."
    )  # 핵심 포인트


class CustomerComplaint(BaseModel):
    """고객 불만 정보를 나타내는 스키마
    
    문제 유형, 심각도, 설명을 구조화된 형태로 추출합니다.
    """

    issue_type: Literal["product", "service", "shipping", "billing"] = Field(
        description="The type of issue"
    )  # 문제 유형
    severity: Literal["low", "medium", "high"] = Field(
        description="The severity of the complaint"
    )  # 심각도
    description: str = Field(
        description="Brief description of the complaint"
    )  # 불만 내용


# 에이전트 생성 - Union 타입으로 여러 스키마 지원
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(Union[ProductReview, CustomerComplaint]),
)

# 리뷰 분석
result1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'",
            }
        ]
    }
)
print("Review:", result1["structured_response"])

# 불만 처리
result2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Customer complaint: Package arrived damaged and contents were broken",
            }
        ]
    }
)
print("Complaint:", result2["structured_response"])

Review: rating=5 sentiment='positive' key_points=['fast shipping', 'expensive', 'great product']
Complaint: issue_type='shipping' severity='high' description='Package arrived damaged and contents were broken'


### 커스텀 도구 메시지 콘텐츠

`tool_message_content` 매개변수를 사용하면 구조화된 출력이 생성될 때 대화 기록에 나타나는 메시지를 커스터마이징할 수 있습니다. 이 메시지는 후속 대화에서 컨텍스트를 제공하는 데 유용하며, 사용자에게 처리 상태를 알려주는 역할도 합니다.

기본적으로 도구 호출 결과는 JSON 형태로 메시지에 기록되지만, 커스텀 메시지를 설정하면 사람이 읽기 쉬운 형태로 표시됩니다. 이는 챗봇이나 대화형 인터페이스에서 사용자 경험을 개선하는 데 도움이 됩니다.

아래 코드는 회의 메모에서 액션 아이템을 추출하고, 커스텀 도구 메시지를 설정하는 예시입니다.

In [7]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """회의에서 추출된 액션 아이템을 나타내는 스키마
    
    담당자, 작업 내용, 우선순위를 구조화합니다.
    """

    task: str = Field(description="The specific task to be completed")  # 작업 내용
    assignee: str = Field(description="Person responsible for the task")  # 담당자
    priority: Literal["low", "medium", "high"] = Field(
        description="Priority level"
    )  # 우선순위


# 에이전트 생성 - 커스텀 도구 메시지 설정
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!",
    ),
)

# 에이전트 실행
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "From our meeting: Sarah needs to update the project timeline as soon as possible",
            }
        ]
    }
)

# 구조화된 응답 출력
print("=== Structured Response ===")
print(result["structured_response"])
# MeetingAction(task='Update the project timeline', assignee='Sarah', priority='high')

# 도구 메시지 콘텐츠 출력
print("=== Tool Message Content ===")
print(result["messages"][-1].content)
# Action item captured and added to meeting notes!

=== Structured Response ===
task='Update the project timeline' assignee='Sarah' priority='high'
=== Tool Message Content ===
Action item captured and added to meeting notes!


---

## 오류 처리

모델은 도구 호출을 통해 구조화된 출력을 생성할 때 스키마와 일치하지 않는 값을 반환할 수 있습니다. 예를 들어, 1-5 범위로 지정된 rating에 10을 반환하거나, 필수 필드를 누락하는 경우가 있습니다. LangChain은 이러한 오류를 자동으로 감지하고 처리하는 지능형 재시도 메커니즘을 제공합니다.

`handle_errors` 매개변수로 오류 처리 방법을 세밀하게 제어할 수 있으며, 기본값은 `True`로 모든 검증 오류를 자동으로 처리합니다. 오류가 발생하면 에이전트는 모델에게 오류 내용을 피드백하고 재시도를 요청하여, 최종적으로 스키마에 맞는 출력을 얻도록 합니다.

이 기능은 프로덕션 환경에서 안정적인 구조화된 출력을 보장하는 데 필수적입니다.

### 스키마 검증 오류

구조화된 출력이 예상 스키마와 일치하지 않으면 에이전트는 오류 피드백을 제공하고 모델에게 재시도를 요청합니다. 예를 들어, rating이 1-5 범위인데 사용자가 "10/10"이라고 입력하면, 모델이 이를 10으로 파싱하려다 검증 오류가 발생하고 자동으로 5로 수정됩니다.

이 재시도 메커니즘 덕분에 실제 사용자 입력이 예상 범위를 벗어나더라도 안정적으로 처리할 수 있습니다. LangSmith에서 실행 추적을 확인하면 재시도 과정을 시각적으로 볼 수 있습니다.

아래 코드는 스키마 검증 오류가 발생했을 때 자동으로 수정되는 예시입니다.

In [8]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductRating(BaseModel):
    """제품 평점 정보를 나타내는 스키마
    
    평점은 1-5 범위로 제한되며, 리뷰 코멘트를 포함합니다.
    """

    rating: int | None = Field(
        description="Rating from 1-5", ge=1, le=5
    )  # 평점 (1-5 범위)
    comment: str = Field(description="Review comment")  # 리뷰 코멘트


# 에이전트 생성 - 기본값 handle_errors=True로 자동 재시도 활성화
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(ProductRating),  # 기본값: handle_errors=True
    system_prompt="You are a helpful assistant that parses product reviews. Do not make any field or value up.",
)

# 10/10 입력 - 범위를 벗어나므로 자동으로 5로 수정됨
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]}
)

# 구조화된 응답 확인
print(result["structured_response"])
# ProductRating(rating=5, comment='Amazing product')
# 모델이 자동으로 수정하여 10을 5로 변경

rating=5 comment='Amazing product, 10/10!'


### 오류 처리 전략

`handle_errors` 매개변수를 사용하여 오류 처리 방법을 다양하게 커스터마이징할 수 있습니다. 요구사항에 따라 자동 재시도, 예외 발생, 커스텀 핸들러 등을 선택할 수 있습니다.

| 설정 | 설명 |
|:---|:---|
| `True` | 모든 오류를 자동으로 처리하고 재시도 (기본값) |
| `False` | 오류 발생 시 예외 발생 |
| 문자열 | 커스텀 오류 메시지로 재시도 |
| 예외 클래스 | 특정 예외만 처리 |
| 콜러블 | 커스텀 오류 핸들러 함수 사용 |

아래에서 각 전략의 사용 예시를 살펴봅니다.

### 커스텀 오류 메시지

문자열을 `handle_errors`에 전달하면 해당 메시지로 모델에게 재시도를 요청합니다. 기본 오류 메시지 대신 더 구체적인 안내를 제공할 수 있으므로, 모델이 오류를 더 잘 이해하고 올바른 형식으로 수정할 수 있습니다.

특히 복잡한 스키마나 특정 도메인의 규칙이 있는 경우, 커스텀 메시지로 명확한 지시를 제공하면 재시도 성공률이 높아집니다.

아래 코드는 커스텀 오류 메시지를 설정하는 예시입니다.

In [9]:
from langchain.agents.structured_output import ToolStrategy

# 커스텀 오류 메시지로 에이전트 생성
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=ProductRating,
        # 오류 발생 시 이 메시지로 재시도 요청
        handle_errors="Please provide a valid rating between 1-5 and include a comment.",
    ),
)

### 특정 예외만 처리

예외 클래스를 `handle_errors`에 전달하면 해당 예외만 처리하고 다른 예외는 그대로 발생합니다. 이는 예상된 검증 오류만 재시도하고, 예상치 못한 시스템 오류는 즉시 감지하고 싶을 때 유용합니다.

예를 들어, `ValueError`만 처리하면 범위 검증 오류는 재시도하지만, 네트워크 오류나 타입 불일치 같은 다른 오류는 예외로 발생시킵니다.

아래 코드는 특정 예외만 처리하는 예시입니다.

In [10]:
# ValueError만 처리하는 에이전트 생성
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=ProductRating,
        handle_errors=ValueError,  # ValueError만 재시도, 다른 예외는 그대로 발생
    ),
)

### 여러 예외 유형 처리

튜플로 여러 예외 클래스를 `handle_errors`에 전달하면 해당 예외들을 모두 처리합니다. 이는 여러 종류의 검증 오류를 한 번에 처리하고 싶을 때 유용합니다.

일반적으로 `ValueError`와 `TypeError`를 함께 처리하면 대부분의 스키마 검증 오류를 커버할 수 있습니다.

아래 코드는 여러 예외 유형을 처리하는 예시입니다.

In [11]:
# 여러 예외 유형을 처리하는 에이전트 생성
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=ProductRating,
        # ValueError 및 TypeError 모두 재시도
        handle_errors=(ValueError, TypeError),
    ),
)

### 커스텀 오류 핸들러 함수

함수(콜러블)를 `handle_errors`에 전달하면 오류 발생 시 해당 함수가 호출되어 커스텀 오류 메시지를 생성합니다. 이 방법은 오류 유형에 따라 다른 메시지를 반환하거나, 오류 정보를 로깅하는 등 복잡한 처리가 필요할 때 유용합니다.

LangChain은 `StructuredOutputValidationError`(스키마 검증 실패)와 `MultipleStructuredOutputsError`(여러 출력 반환) 등의 구체적인 예외 클래스를 제공합니다. 이를 활용하면 오류 유형별로 세밀한 처리가 가능합니다.

아래 코드는 커스텀 오류 핸들러 함수를 사용하는 예시입니다.

In [12]:
from langchain.agents.structured_output import (
    StructuredOutputValidationError,
    MultipleStructuredOutputsError,
)


def custom_error_handler(error: Exception) -> str:
    """오류 유형에 따른 커스텀 오류 메시지 생성
    
    Args:
        error: 발생한 예외 객체
        
    Returns:
        모델에게 전달할 오류 메시지
    """
    if isinstance(error, StructuredOutputValidationError):
        # 스키마 검증 오류
        return "There was an issue with the format. Try again."
    elif isinstance(error, MultipleStructuredOutputsError):
        # 여러 구조화된 출력이 반환된 경우
        return "Multiple structured outputs were returned. Pick the most relevant one."
    else:
        # 기타 오류
        return f"Error: {str(error)}"


# 커스텀 오류 핸들러로 에이전트 생성
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=ProductRating, handle_errors=custom_error_handler
    ),
)

### 오류 처리 비활성화

`handle_errors=False`를 설정하면 오류 발생 시 예외가 그대로 발생합니다. 이는 개발 중에 오류를 디버깅하거나, 커스텀 오류 처리 로직을 직접 구현하고 싶을 때 유용합니다.

프로덕션 환경에서는 일반적으로 `handle_errors=True`(기본값)를 사용하여 안정적인 동작을 보장하지만, 개발 및 테스트 환경에서는 비활성화하여 문제를 빠르게 파악할 수 있습니다.

아래 코드는 오류 처리를 비활성화하는 예시입니다.

In [13]:
# 오류 처리 비활성화 - 모든 오류가 예외로 발생
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=ProductRating, handle_errors=False  # 오류 발생 시 예외 발생
    ),
)

---

## 종합 예제

지금까지 학습한 `Union` 타입과 오류 처리를 결합한 실용적인 예제입니다. 하나의 에이전트가 책과 영화 추천을 모두 처리하며, 입력 내용에 따라 적절한 스키마를 자동으로 선택합니다.

`handle_errors=True`로 설정하여 오류 발생 시 자동으로 재시도하므로, 프로덕션 환경에서도 안정적으로 동작합니다. 이 패턴은 다양한 유형의 입력을 처리해야 하는 챗봇이나 AI 어시스턴트에 적합합니다.

아래 코드는 책과 영화 추천을 처리하는 에이전트 예시입니다.

In [14]:
from pydantic import BaseModel, Field
from typing import Literal, Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class BookRecommendation(BaseModel):
    """책 추천 정보를 나타내는 스키마
    
    제목, 저자, 장르, 평점, 요약을 구조화합니다.
    """

    title: str = Field(description="Book title")  # 책 제목
    author: str = Field(description="Author name")  # 저자명
    genre: Literal["fiction", "non-fiction", "science", "history", "biography"] = Field(
        description="Book genre"
    )  # 장르
    rating: int = Field(description="Rating from 1-5", ge=1, le=5)  # 평점 (1-5)
    summary: str = Field(description="Brief summary of the book")  # 책 요약


class MovieRecommendation(BaseModel):
    """영화 추천 정보를 나타내는 스키마
    
    제목, 감독, 개봉년도, 장르, 평점을 구조화합니다.
    """

    title: str = Field(description="Movie title")  # 영화 제목
    director: str = Field(description="Director name")  # 감독명
    year: int = Field(description="Release year")  # 개봉년도
    genre: Literal["action", "comedy", "drama", "horror", "sci-fi"] = Field(
        description="Movie genre"
    )  # 장르
    rating: int = Field(description="Rating from 1-5", ge=1, le=5)  # 평점 (1-5)


# 에이전트 생성 - Union 타입으로 여러 추천 유형 지원
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=Union[BookRecommendation, MovieRecommendation], handle_errors=True
    ),
    system_prompt="You are a helpful entertainment recommendation assistant.",
)

# 책 추천 요청
result1 = agent.invoke(
    {"messages": [{"role": "user", "content": "Recommend a good science fiction book"}]}
)
print("Book recommendation:")
print(result1["structured_response"])

# 영화 추천 요청
result2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Recommend a comedy movie from the 2000s"}
        ]
    }
)
print("\nMovie recommendation:")
print(result2["structured_response"])

Book recommendation:
title='Dune' author='Frank Herbert' genre='science' rating=5 summary='Set in the distant future, Dune follows Paul Atreides as his family takes control of the desert planet Arrakis, the only source of the valuable spice melange. The story weaves together themes of politics, religion, ecology, and destiny as Paul navigates betrayal, finds his place among the native Fremen, and potentially fulfills an ancient prophecy. This epic tale explores power, survival, and human potential in a richly imagined universe.'

Movie recommendation:
title='Superbad' director='Greg Mottola' year=2007 genre='comedy' rating=4


---

## 정리

이 튜토리얼에서는 LangGraph 에이전트의 구조화된 출력 기능을 학습했습니다. 구조화된 출력을 사용하면 에이전트의 응답을 예측 가능한 형식으로 받아 애플리케이션에서 쉽게 처리할 수 있습니다.

**핵심 개념 요약:**

| 개념 | 설명 |
|:---|:---|
| **ProviderStrategy** | OpenAI, Anthropic, Grok 등 네이티브 지원 모델용 (가장 신뢰성 높음) |
| **ToolStrategy** | 도구 호출을 통한 구조화된 출력 (대부분의 모델 지원) |
| **Union 타입** | 여러 스키마 중 자동 선택 |
| **handle_errors** | 오류 발생 시 자동 재시도 및 수정 |

**스키마 정의 방법:**
- **Pydantic 모델**: 가장 풍부한 검증 기능 제공 (권장)
- **데이터클래스**: 간단한 스키마 정의, 추가 의존성 없음
- **TypedDict**: 딕셔너리 형태로 반환, JSON 직렬화 용이

**실전 팁:**
- 필드에 명확한 `description` 제공
- `Literal` 타입으로 허용값 제한
- `ge`, `le` 등 검증자로 범위 지정
- `Union` 타입으로 다양한 응답 유형 처리
- 프로덕션에서는 `handle_errors=True` 사용 (기본값)